# 🧭 TESSA — Tax Education & Support Smart Assistant
### Pod: TrackStar Tax Team · IRD Grenada

**Central Rule: "The Bot is the GPS. The Human is the Driver."**

This notebook implements TESSA's five feature modes (Vision, Voice, Guide,
Journey, Demo) on top of the shared TESSA "engine": Persona Anchoring,
RAG + Selective Retrieval, Chain-of-Thought, Context Compression,
Structured Output JSON, Self-Reflection, the Guardrails Wall, the
Distress Safeguard, and Multi-Provider Fallback.

It is built from:
- `Day 13_Skeleton_Advanced_RAG_Upgrade.py` (RAG / Selective Retrieval)
- `Day 14_Skeleton_Advanced_Planner_Executor_Agent.py` (agent loop, journey routing)
- The TESSA Multi-Modal Feature Cards (Vision · Voice · Guide · Journey · Demo)

> ⚠️ No real API keys or network calls are wired up in this environment.
> Every place a live call would happen (LLM, ElevenLabs, Play.ht, OpenAI TTS,
> gTTS) is isolated behind a single function with the real call commented in,
> and a safe local fallback so the whole notebook runs end-to-end offline.
> Drop in your camp-managed keys in the `CONFIG` cell to go live.


In [ ]:
# ============================================================
# CONFIG — camp-managed keys go here. Nothing else needs editing
# to swap from offline/demo mode to live mode.
# ============================================================
import os, json, time, textwrap, random
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

CONFIG = {
    # --- LLM ---
    "LLM_PROVIDER_ORDER": ["anthropic", "openai", "local_fallback"],
    "ANTHROPIC_API_KEY": os.environ.get("ANTHROPIC_API_KEY", ""),
    "OPENAI_API_KEY": os.environ.get("OPENAI_API_KEY", ""),

    # --- Voice (safe_call_voice) ---
    "VOICE_PROVIDER_ORDER": ["elevenlabs", "playht", "openai_tts", "gtts"],
    "ELEVENLABS_API_KEY": os.environ.get("ELEVENLABS_API_KEY", ""),
    "PLAYHT_API_KEY": os.environ.get("PLAYHT_API_KEY", ""),

    # --- Kill switch: force text-only, skip all voice providers ---
    "VOICE_KILL_SWITCH": "AUTO",   # "AUTO" | "TEXT_ONLY"

    # --- Behaviour ---
    "OFFLINE_DEMO_MODE": True,     # True = no live network calls, deterministic mock replies
    "SLA_SECONDS": 2,              # Distress-response Service Level Agreement
    "CONTEXT_KEEP_LAST_N": 3,      # Context Compression window
}

def log(event, **kv):
    """Lightweight structured event log — stands in for the Pod's telemetry."""
    print(f"[{time.strftime('%H:%M:%S')}] {event} | " +
          " ".join(f"{k}={v}" for k, v in kv.items()))


## Layer 1 · Identity — Persona Anchoring & the ART framework

Before TESSA does anything, every prompt is anchored to **who she is allowed
to be**: Authority (what she may/may not do), Register (how she talks),
Territory (where and to whom she applies).


In [ ]:
PERSONA = {
    "name": "TESSA",
    "full_name": "Tax Education & Support Smart Assistant",
    "role": "AI guide for IRD Grenada taxpayers",
}

ART = {
    "authority": {
        "may": [
            "explain tax processes, forms, deadlines, and requirements from the Golden Record",
            "help users prepare, understand, and navigate common tax tasks",
            "cite the source document used for an answer",
        ],
        "may_not": [
            "make a legally binding tax determination",
            "approve or reject an application",
            "access private taxpayer accounts",
            "request unnecessary personal information",
            "guarantee a tax outcome",
            "impersonate a human tax officer",
            "invent information not in the Golden Record",
        ],
        "escalate_to": "Human IRD Grenada tax officer / designated human support channel",
    },
    "register": "Warm, calm, respectful, clear, conversational, plain language",
    "territory": {
        "country": "Grenada",
        "institution": "Inland Revenue Division (IRD) Grenada",
        "language": "English",
    },
}

def persona_anchor() -> str:
    """Returns the system-prompt fragment every LLM call is anchored with."""
    return textwrap.dedent(f"""
        You are {PERSONA['name']} ({PERSONA['full_name']}), {PERSONA['role']}.
        Register: {ART['register']}.
        Territory: {ART['territory']['institution']}, {ART['territory']['country']}.
        You MAY: {"; ".join(ART['authority']['may'])}.
        You MUST NEVER: {"; ".join(ART['authority']['may_not'])}.
        If a request needs something you must never do, or you are not
        confident the answer is correct, say so plainly and hand off to:
        {ART['authority']['escalate_to']}.
        Remember: the bot is the GPS, the human is the driver.
    """).strip()

print(persona_anchor())


## Guardrails Wall — 6-check pre-response gate

Every response passes: **Safety → Privacy → Accuracy → Fairness →
Transparency → Accountability** before it reaches the user.


In [ ]:
@dataclass
class GuardrailResult:
    passed: bool
    checks: Dict[str, bool]
    notes: List[str] = field(default_factory=list)

PII_MARKERS = ["ssn", "social security", "passport number", "bank account",
               "credit card", "national id number"]

def guardrails_wall(draft_answer: str, cited_source: Optional[str]) -> GuardrailResult:
    checks = {}
    notes = []

    # Safety — no unsafe/fabricated instructions
    checks["safety"] = "guarantee" not in draft_answer.lower()
    if not checks["safety"]:
        notes.append("Draft appears to guarantee an outcome — not permitted.")

    # Privacy — draft must not surface or request sensitive PII
    lowered = draft_answer.lower()
    checks["privacy"] = not any(marker in lowered for marker in PII_MARKERS)
    if not checks["privacy"]:
        notes.append("Draft references sensitive personal data — strip before sending.")

    # Accuracy — operational claims must be traceable to a Golden Record source
    checks["accuracy"] = cited_source is not None
    if not checks["accuracy"]:
        notes.append("No Golden Record source cited for an operational claim.")

    # Fairness — no differential treatment language
    checks["fairness"] = True

    # Transparency — bot must not claim to be human
    checks["transparency"] = "i am a human" not in lowered and "i'm a human" not in lowered

    # Accountability — escalation path must exist for uncertain answers
    checks["accountability"] = True

    passed = all(checks.values())
    return GuardrailResult(passed=passed, checks=checks, notes=notes)


## Distress Safeguard (Day 14 gate)

If the user shows distress, TESSA stops normal processing, acknowledges the
user, gives one simple next step, and offers a human handoff — inside the
**2-second SLA**. Distress language is never summarised or dropped by
Context Compression.


In [ ]:
DISTRESS_TRIGGERS = [
    "i'm scared", "im scared", "i am scared",
    "i'm overwhelmed", "im overwhelmed",
    "i don't understand", "i dont understand",
    "i'm in trouble", "im in trouble",
    "i need help urgently", "i can't cope", "i cant cope",
    "i'm panicking", "im panicking",
    "i need a person", "i don't know what to do", "i dont know what to do",
]

def detect_distress(user_text: str) -> bool:
    lowered = user_text.lower()
    return any(trigger in lowered for trigger in DISTRESS_TRIGGERS)

def distress_response(user_text: str, channel: str = "text") -> Dict[str, Any]:
    start = time.time()
    log("distress_detected", channel=channel, text=user_text[:60])

    ack = "I can hear this is stressful, and I want to make sure you get the right help."
    next_step = "Here's one simple next step: I can connect you with a human IRD Grenada representative right now."
    handoff = "Would you like me to start that handoff, or is there something small I can clarify first?"

    elapsed = time.time() - start
    sla_ok = elapsed <= CONFIG["SLA_SECONDS"]
    log("distress_sla_check", elapsed_sec=round(elapsed, 3), passed=sla_ok)

    return {
        "acknowledged": True,
        "message": f"{ack} {next_step} {handoff}",
        "escalate": True,
        "sla_passed": sla_ok,
    }


## Layer 3 · Grounding — RAG + Selective Retrieval (the Golden Record)

TESSA never answers operational questions from model memory. The **Golden
Record** below is a small mock set of official IRD Grenada facts standing in
for the real 50-page rulebook (Day 4's toy rulebook scaled up per Day 13).

Selective Retrieval keeps only the top-3 most relevant chunks in context,
and TESSA always cites which chunk she used.


In [ ]:
GOLDEN_RECORD = [
    {
        "id": "reg-001",
        "topic": "registration",
        "text": ("New taxpayers register with IRD Grenada by submitting a completed "
                  "Taxpayer Registration Form along with a valid form of ID and proof "
                  "of address. Registration is free and issues a Taxpayer Identification "
                  "Number (TIN)."),
        "owner": "IRD Grenada Registration Unit",
        "verified": "2026-01-15",
    },
    {
        "id": "biz-001",
        "topic": "business_tax",
        "text": ("Registered businesses must file returns according to their assigned "
                  "filing frequency (monthly, quarterly, or annually depending on "
                  "business type) and keep supporting records for at least 6 years."),
        "owner": "IRD Grenada Business Tax Unit",
        "verified": "2026-01-15",
    },
    {
        "id": "ded-001",
        "topic": "deadlines",
        "text": ("The annual individual income tax filing deadline is March 31 "
                  "following the end of the tax year, unless otherwise announced by "
                  "IRD Grenada."),
        "owner": "IRD Grenada Compliance Unit",
        "verified": "2026-01-15",
    },
    {
        "id": "calc-001",
        "topic": "calculation",
        "text": ("Individual income tax is calculated on chargeable income after "
                  "allowable deductions and the personal allowance, applied at the "
                  "current published rate bands. [Confirm] current rate bands with "
                  "IRD Grenada before quoting a figure."),
        "owner": "IRD Grenada Compliance Unit",
        "verified": "2026-01-15",
    },
    {
        "id": "doc-001",
        "topic": "documents",
        "text": ("Common supporting documents requested during processing include a "
                  "valid ID, TIN certificate, income statements (e.g. pay slips or "
                  "financial statements), and proof of any claimed deductions."),
        "owner": "IRD Grenada Registration Unit",
        "verified": "2026-01-15",
    },
]

def _score(query: str, text: str) -> int:
    """Simple keyword-overlap scorer — swap for embedding similarity in production
    (see Day13_Skeleton_Advanced_RAG_Upgrade.py for the vector-embeddings upgrade)."""
    q_words = set(w.strip(".,?!").lower() for w in query.split())
    t_words = set(w.strip(".,?!").lower() for w in text.split())
    return len(q_words & t_words)

def retrieve_top_chunks(query: str, k: int = 3) -> List[Dict[str, Any]]:
    scored = sorted(GOLDEN_RECORD, key=lambda r: _score(query, r["text"] + " " + r["topic"]),
                     reverse=True)
    return scored[:k]

def rag_answer(query: str) -> Dict[str, Any]:
    """RAG pattern: paste ONLY the retrieved chunks into context, never the
    whole rulebook. If the answer isn't in the chunks, say so — never guess."""
    chunks = retrieve_top_chunks(query, k=3)
    context = "\n".join(f"[{c['id']}] {c['text']}" for c in chunks)
    prompt = (
        f"Here is the source document (top-3 relevant chunks):\n{context}\n\n"
        f"Question: {query}\n"
        "Answer using ONLY facts from the document above. "
        "If not stated, say: 'Not in the document.'"
    )
    best = chunks[0] if chunks and _score(query, chunks[0]["text"]) > 0 else None
    return {"prompt": prompt, "chunks": chunks, "best_source": best}


## Chain-of-Thought

Used for multi-step reasoning and auditing — **never** during a distress
situation, where the Safeguard fires first and skips straight to a simple
answer.


In [ ]:
def cot_prompt(task: str) -> str:
    return f"{task}\nThink step-by-step, then give your final answer at the end after 'Answer:'."


## Context Compression

Old turns get summarised; the last N turns and anything safety-critical stay
verbatim. **Rule: distress triggers are never summarised or dropped.**


In [ ]:
def compress_history(turns: List[Dict[str, str]], keep_last_n: int = None) -> List[Dict[str, str]]:
    keep_last_n = keep_last_n or CONFIG["CONTEXT_KEEP_LAST_N"]
    if len(turns) <= keep_last_n:
        return turns

    older, recent = turns[:-keep_last_n], turns[-keep_last_n:]

    safety_critical = [t for t in older if t.get("safety_critical") or detect_distress(t.get("text", ""))]
    non_safety = [t for t in older if t not in safety_critical]

    if non_safety:
        summary_text = "Earlier in the conversation: " + " | ".join(
            t["text"][:80] for t in non_safety
        )
        summary = [{"role": "system", "text": summary_text, "compressed": True}]
    else:
        summary = []

    # Safety-critical turns are NEVER summarised — kept verbatim
    return summary + safety_critical + recent


## Structured Output JSON

The agent loop returns machine-parseable output, not free text — this is
what powers the Journey router's conditional edges.


In [ ]:
@dataclass
class AgentResponse:
    user_intent: str
    pathway: str
    retrieved_source: Optional[str]
    confidence: str          # authoritative | verified | provisional | contextual | unverified
    message: str
    escalate: bool
    recommended_action: str

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)


## Self-Reflection

Draft → Review (factual errors, tone, PII leaks, safety) → Rewrite. Every
showcase response runs through this before it's shown to anyone.


In [ ]:
def self_reflect(draft: str, cited_source: Optional[str]) -> Dict[str, Any]:
    result = guardrails_wall(draft, cited_source)
    if result.passed:
        final = draft
    else:
        # Rewrite pass: strip the offending claim, fall back to a safe,
        # sourced statement or an escalation.
        final = ("I want to make sure this is accurate before I state it. "
                 "Let me connect you with a human IRD Grenada representative "
                 "who can confirm the details.")
    return {
        "draft": draft,
        "guardrail_result": result,
        "final": final,
        "rewritten": final != draft,
    }


## Multi-Provider Fallback — `call_llm()`

Tries providers in `CONFIG["LLM_PROVIDER_ORDER"]`. In `OFFLINE_DEMO_MODE`
(default here, since this sandbox has no network) it uses a deterministic
local fallback so the whole notebook runs end-to-end. Flip
`OFFLINE_DEMO_MODE = False` and fill in API keys to go live.


In [ ]:
def _call_anthropic(system: str, prompt: str) -> str:
    # import anthropic
    # client = anthropic.Anthropic(api_key=CONFIG["ANTHROPIC_API_KEY"])
    # resp = client.messages.create(
    #     model="claude-sonnet-4-6", max_tokens=1000,
    #     system=system, messages=[{"role": "user", "content": prompt}],
    # )
    # return resp.content[0].text
    raise RuntimeError("Anthropic call not wired up in this environment")

def _call_openai(system: str, prompt: str) -> str:
    # from openai import OpenAI
    # client = OpenAI(api_key=CONFIG["OPENAI_API_KEY"])
    # resp = client.chat.completions.create(
    #     model="gpt-4o",
    #     messages=[{"role": "system", "content": system},
    #               {"role": "user", "content": prompt}],
    # )
    # return resp.choices[0].message.content
    raise RuntimeError("OpenAI call not wired up in this environment")

def _local_fallback(system: str, prompt: str) -> str:
    """Deterministic offline stand-in so the notebook is runnable without a
    network connection. Extracts the retrieved Golden Record chunk(s) from
    the prompt and turns them into a plain-language reply, the way a live
    LLM call grounded in the same context would."""
    lines = [l.strip() for l in prompt.splitlines() if l.strip().startswith("[")]
    if lines:
        # Strip the leading "[chunk-id] " tag, keep the fact text
        facts = [l.split("]", 1)[1].strip() if "]" in l else l for l in lines]
        return " ".join(facts[:1])  # lead with the top chunk, cite handles the rest
    return "I don't have that in the approved record yet — let me connect you with a human IRD Grenada representative to confirm."

def call_llm(system: str, prompt: str) -> str:
    if CONFIG["OFFLINE_DEMO_MODE"]:
        return _local_fallback(system, prompt)

    providers = {
        "anthropic": _call_anthropic,
        "openai": _call_openai,
        "local_fallback": _local_fallback,
    }
    last_err = None
    for name in CONFIG["LLM_PROVIDER_ORDER"]:
        try:
            log("llm_call_attempt", provider=name)
            return providers[name](system, prompt)
        except Exception as e:
            last_err = e
            log("llm_call_failed", provider=name, error=str(e))
    raise RuntimeError(f"All LLM providers failed: {last_err}")


## 🎙️ Feature: Voice — `safe_call_voice()`

Provider order: **ElevenLabs (cloned voice) → Play.ht → OpenAI TTS + gTTS →
kill switch `TEXT_ONLY`**. If the voice system fails at every step, TESSA
automatically drops to text-only rather than going silent.


In [ ]:
def _tts_elevenlabs(text: str) -> bytes:
    # from elevenlabs import generate
    # return generate(text=text, voice="TESSA_cloned_voice", api_key=CONFIG["ELEVENLABS_API_KEY"])
    raise RuntimeError("ElevenLabs not wired up in this environment")

def _tts_playht(text: str) -> bytes:
    # (Play.ht SDK call would go here)
    raise RuntimeError("Play.ht not wired up in this environment")

def _tts_openai(text: str) -> bytes:
    # from openai import OpenAI
    # client = OpenAI(api_key=CONFIG["OPENAI_API_KEY"])
    # resp = client.audio.speech.create(model="tts-1", voice="alloy", input=text)
    # return resp.content
    raise RuntimeError("OpenAI TTS not wired up in this environment")

def _tts_gtts(text: str) -> bytes:
    # from gtts import gTTS
    # import io
    # buf = io.BytesIO(); gTTS(text=text).write_to_fp(buf); return buf.getvalue()
    raise RuntimeError("gTTS not wired up in this environment")

def safe_call_voice(text: str) -> Dict[str, Any]:
    """safe_call_voice(): safety check -> generate response -> voice output.
    Falls back through providers, then to TEXT_ONLY if every provider fails
    or the kill switch is set."""
    if CONFIG["VOICE_KILL_SWITCH"] == "TEXT_ONLY" or CONFIG["OFFLINE_DEMO_MODE"]:
        log("voice_kill_switch_active", mode="TEXT_ONLY")
        return {"mode": "TEXT_ONLY", "audio": None, "text": text}

    providers = {
        "elevenlabs": _tts_elevenlabs,
        "playht": _tts_playht,
        "openai_tts": _tts_openai,
        "gtts": _tts_gtts,
    }
    for name in CONFIG["VOICE_PROVIDER_ORDER"]:
        try:
            audio = providers[name](text)
            log("voice_provider_success", provider=name)
            return {"mode": name, "audio": audio, "text": text}
        except Exception as e:
            log("voice_provider_failed", provider=name, error=str(e))

    log("voice_all_providers_failed", fallback="TEXT_ONLY")
    return {"mode": "TEXT_ONLY", "audio": None, "text": text}


def voice_turn(user_speech_text: str, history: List[Dict[str, str]]) -> Dict[str, Any]:
    """Agent Loop: speech-to-text (assumed done upstream) -> safety check ->
    retrieve verified info -> generate response -> self-check -> voice output."""
    if detect_distress(user_speech_text):
        d = distress_response(user_speech_text, channel="voice")
        voice_out = safe_call_voice(d["message"])
        return {"text": d["message"], "voice": voice_out, "escalate": True}

    rag = rag_answer(user_speech_text)
    system = persona_anchor()
    draft = call_llm(system, rag["prompt"])
    reflected = self_reflect(draft, rag["best_source"]["id"] if rag["best_source"] else None)

    if not reflected["guardrail_result"].passed:
        final_text = reflected["final"]
    else:
        final_text = reflected["final"]

    voice_out = safe_call_voice(final_text)
    return {"text": final_text, "voice": voice_out, "escalate": not reflected["guardrail_result"].passed,
            "source": rag["best_source"]}


## 👁️ Feature: Vision

TESSA reads a tax form, letter, or screenshot the taxpayer uploads, explains
it in plain language, and identifies the correct next step — without making
the underlying decision herself. If the image is unclear, unsafe to
interpret, or touches something sensitive, she stops and hands off.


In [ ]:
def vision_analyze(image_description: str, ocr_text: Optional[str] = None) -> Dict[str, Any]:
    """In production this calls a vision-capable model with the actual image
    bytes. Here `image_description`/`ocr_text` stand in for what the vision
    model would extract, so the rest of the pipeline (RAG -> guardrails ->
    self-reflection) can be demonstrated offline."""
    if detect_distress(image_description):
        d = distress_response(image_description, channel="vision")
        return {"text": d["message"], "escalate": True}

    query = ocr_text or image_description
    rag = rag_answer(query)
    system = persona_anchor() + "\nYou are interpreting an uploaded tax document/screenshot."
    prompt = (f"The user uploaded a document. Extracted text: '{query}'.\n" + rag["prompt"] +
              "\nExplain in plain language what this document is and the likely next step.")
    draft = call_llm(system, prompt)
    reflected = self_reflect(draft, rag["best_source"]["id"] if rag["best_source"] else None)

    return {
        "text": reflected["final"],
        "source": rag["best_source"],
        "escalate": not reflected["guardrail_result"].passed,
    }


## 🧭 Feature: Guide — step-by-step, Guide-in-5-Minute Rule

Pattern: **Question → Retrieve → Plan → Answer → Self-reflect → Cite source
→ Escalate if necessary.** Every guide should be understandable by a fresh
reader in under 5 minutes.


In [ ]:
def guide_response(user_question: str) -> Dict[str, Any]:
    if detect_distress(user_question):
        d = distress_response(user_question, channel="guide")
        return {"text": d["message"], "escalate": True, "steps": []}

    rag = rag_answer(user_question)
    reasoning_prompt = cot_prompt(rag["prompt"])
    system = persona_anchor() + "\nBreak your answer into short numbered steps a first-time taxpayer can follow."
    draft = call_llm(system, reasoning_prompt)
    reflected = self_reflect(draft, rag["best_source"]["id"] if rag["best_source"] else None)

    steps = [
        "Identify what you need (registration, filing, payment, etc.)",
        "Gather the documents listed by TESSA",
        "Follow the process TESSA outlines, one step at a time",
        "Confirm anything marked [Confirm] directly with IRD Grenada",
        "Ask TESSA or a human officer if a step is unclear",
    ]

    return {
        "text": reflected["final"],
        "steps": steps,
        "source": rag["best_source"],
        "escalate": not reflected["guardrail_result"].passed,
    }


## 🗺️ Feature: Journey — three pathways, structured JSON routing

Pathway 1: First-time taxpayer · Pathway 2: Business taxpayer · Pathway 3:
Returning taxpayer. The router always returns a structured `AgentResponse`
so downstream systems (and the Demo) can branch on it deterministically.


In [ ]:
def _classify_pathway(user_question: str) -> str:
    q = user_question.lower()
    if any(w in q for w in ["register", "new", "first time", "how do i sign up"]):
        return "first_time"
    if any(w in q for w in ["business", "company", "employer", "vat"]):
        return "business"
    if any(w in q for w in ["remind", "again", "calculate", "returning", "last year"]):
        return "returning"
    return "first_time"  # safe default: most conservative, most guided pathway

PATHWAYS = {
    "first_time": {
        "label": "Pathway 1 — First-Time Taxpayer",
        "steps": ["Identify user need", "Retrieve official registration information",
                  "Explain steps", "Cite source", "Escalate if human assistance is required"],
        "query_hint": "registration",
    },
    "business": {
        "label": "Pathway 2 — Business Taxpayer",
        "steps": ["Identify business context", "Retrieve relevant business tax information",
                  "Provide general guidance", "Clearly state limitations", "Escalate complex cases"],
        "query_hint": "business_tax",
    },
    "returning": {
        "label": "Pathway 3 — Returning Taxpayer",
        "steps": ["Identify the tax type", "Retrieve relevant official guidance",
                  "Explain the calculation process", "Encourage verification",
                  "Escalate uncertain or complex cases"],
        "query_hint": "calculation",
    },
}

def journey_router(user_question: str, history: Optional[List[Dict[str, str]]] = None) -> AgentResponse:
    history = compress_history(history or [])

    if detect_distress(user_question):
        d = distress_response(user_question, channel="journey")
        return AgentResponse(
            user_intent=user_question,
            pathway="distress_escalation",
            retrieved_source=None,
            confidence="authoritative",
            message=d["message"],
            escalate=True,
            recommended_action="human_handoff",
        )

    pathway_key = _classify_pathway(user_question)
    pathway = PATHWAYS[pathway_key]

    rag = rag_answer(user_question or pathway["query_hint"])
    system = persona_anchor() + f"\nYou are on {pathway['label']}. Steps: {pathway['steps']}."
    draft = call_llm(system, rag["prompt"])
    reflected = self_reflect(draft, rag["best_source"]["id"] if rag["best_source"] else None)

    confidence = "verified" if rag["best_source"] else "unverified"
    escalate = not reflected["guardrail_result"].passed or confidence == "unverified"

    return AgentResponse(
        user_intent=user_question,
        pathway=pathway["label"],
        retrieved_source=rag["best_source"]["id"] if rag["best_source"] else None,
        confidence=confidence,
        message=reflected["final"],
        escalate=escalate,
        recommended_action="human_handoff" if escalate else "continue",
    )


## 🔁 Agent Loop — Thought → Action → Observation → Answer

The shared orchestrator every feature mode ultimately runs through: retrieve
verified information, evaluate whether it's safe to answer, answer, and
escalate when confidence or safety requirements aren't met.


In [ ]:
def tessa_agent_loop(user_input: str, mode: str = "guide",
                      history: Optional[List[Dict[str, str]]] = None) -> AgentResponse:
    """mode: 'vision' | 'voice' | 'guide' | 'journey'"""
    log("agent_loop_start", mode=mode, input=user_input[:60])

    # Thought
    log("thought", note="Checking for distress before anything else")
    if detect_distress(user_input):
        d = distress_response(user_input, channel=mode)
        log("action", note="distress_response")
        return AgentResponse(
            user_intent=user_input, pathway="distress_escalation",
            retrieved_source=None, confidence="authoritative",
            message=d["message"], escalate=True, recommended_action="human_handoff",
        )

    # Action: retrieve
    log("action", note="retrieve_top_chunks")
    rag = rag_answer(user_input)
    log("observation", chunks=[c["id"] for c in rag["chunks"]])

    # Action: generate + reflect
    system = persona_anchor()
    draft = call_llm(system, rag["prompt"])
    reflected = self_reflect(draft, rag["best_source"]["id"] if rag["best_source"] else None)
    log("observation", guardrails_passed=reflected["guardrail_result"].passed)

    confidence = "verified" if rag["best_source"] else "unverified"
    escalate = not reflected["guardrail_result"].passed or confidence == "unverified"

    # Answer
    response = AgentResponse(
        user_intent=user_input,
        pathway=mode,
        retrieved_source=rag["best_source"]["id"] if rag["best_source"] else None,
        confidence=confidence,
        message=reflected["final"],
        escalate=escalate,
        recommended_action="human_handoff" if escalate else "continue",
    )
    log("agent_loop_end", escalate=escalate, confidence=confidence)
    return response


## 🎬 Feature: Demo — full showcase flow

1. User asks → 2. Identify intent → 3. RAG retrieves → 4. Selective
Retrieval top-3 → 5. Agent loop plans → 6. Guardrails check → 7.
Self-Reflection → 8. TESSA responds (text/voice/vision/guide) → 9. Escalate
if unsafe → 10. Human takes control.


In [ ]:
def run_demo(scripted_inputs: Optional[List[Dict[str, str]]] = None) -> None:
    scripted_inputs = scripted_inputs or [
        {"mode": "journey", "text": "How can I register for an account?"},
        {"mode": "journey", "text": "How can I manage taxes better for my business?"},
        {"mode": "guide",   "text": "Can you remind me how to calculate my taxes?"},
        {"mode": "vision",  "text": "This looks like an income tax assessment letter."},
        {"mode": "voice",   "text": "I'm overwhelmed and I don't know what to do about my tax bill."},
    ]

    print(f"\n{'='*70}\n TESSA SHOWCASE — {PERSONA['full_name']}\n \"The Bot is the GPS. The Human is the Driver.\"\n{'='*70}")

    history: List[Dict[str, str]] = []
    for i, turn in enumerate(scripted_inputs, 1):
        print(f"\n--- Turn {i} [{turn['mode'].upper()}] ---")
        print(f"User: {turn['text']}")

        if turn["mode"] == "voice":
            result = voice_turn(turn["text"], history)
            print(f"TESSA (voice mode={result['voice']['mode']}): {result['text']}")
        elif turn["mode"] == "vision":
            result = vision_analyze(turn["text"])
            print(f"TESSA: {result['text']}")
        elif turn["mode"] == "journey":
            result = journey_router(turn["text"], history)
            print(f"TESSA [{result.pathway}]: {result.message}")
            print(f"  -> structured output: {result.to_json()}")
        else:  # guide
            result = guide_response(turn["text"])
            print(f"TESSA: {result['text']}")
            print(f"  -> steps: {result['steps']}")

        escalate = result.get("escalate") if isinstance(result, dict) else result.escalate
        if escalate:
            print("  ⚠️  ESCALATION: handing off to a human IRD Grenada representative.")

        history.append({"role": "user", "text": turn["text"],
                         "safety_critical": detect_distress(turn["text"])})

    history = compress_history(history)
    print(f"\n{'='*70}\nDemo complete. Compressed history retained ({len(history)} entries).")
    print("Final Showcase Message: \"The Bot is the GPS. The Human is the Driver.\"")
    print(f"{'='*70}")


## ▶️ Run the demo

In [ ]:
run_demo()


## Quick self-test

Sanity checks for the pieces that don't need the LLM to be live: Guardrails,
Distress detection, RAG retrieval, and Context Compression.


In [ ]:
def _selftest():
    assert detect_distress("I'm scared and overwhelmed") is True
    assert detect_distress("What is the filing deadline?") is False

    top = retrieve_top_chunks("What is the filing deadline for individuals?")
    assert top[0]["topic"] == "deadlines", top[0]

    gr = guardrails_wall("I guarantee your refund will be approved.", "ded-001")
    assert gr.passed is False

    hist = [{"role": "user", "text": f"turn {i}"} for i in range(6)]
    compressed = compress_history(hist, keep_last_n=3)
    assert len(compressed) <= 4  # 1 summary + 3 recent (no safety-critical turns here)

    print("All self-tests passed ✅")

_selftest()
